# Lab 8: LoRA Fine-Tuning for Text-to-SQL (Olist Dataset)

This notebook fine-tunes **CodeLlama-7B-Instruct** using **LoRA (QLoRA 4-bit)** on the Olist Text-to-SQL instruction dataset.

**Requirements:** Colab with T4 GPU (free tier works)

**Steps:**
1. Install dependencies
2. Upload instruction dataset
3. Load base model with 4-bit quantization
4. Apply LoRA and train
5. Download the fine-tuned adapter

## 1. Install Dependencies

In [ ]:
!pip install -q torch transformers peft datasets accelerate bitsandbytes

## 2. Upload Instruction Dataset

Upload `data/instruction_dataset.json` from your repo.
Run the cell below — it will open a file picker.

In [ ]:
from google.colab import files
import json

uploaded = files.upload()  # Upload instruction_dataset.json

filename = list(uploaded.keys())[0]
with open(filename) as f:
    dataset_raw = json.load(f)

print(f"Loaded {len(dataset_raw)} examples")
print(f"Sample: {dataset_raw[0]['input'][:80]}...")

## 3. Prepare Dataset

In [ ]:
import random
from datasets import Dataset

random.seed(42)
random.shuffle(dataset_raw)

split = int(len(dataset_raw) * 0.9)
train_data = dataset_raw[:split]
val_data = dataset_raw[split:]

train_ds = Dataset.from_list(train_data)
val_ds = Dataset.from_list(val_data)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

## 4. Load Base Model (QLoRA 4-bit)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "codellama/CodeLlama-7b-Instruct-hf"

# 4-bit quantization config (fits T4 16GB)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    trust_remote_code=True,
)

print("Model loaded!")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB")

## 5. Apply LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Tokenize Data

In [ ]:
MAX_LENGTH = 1024

def format_example(example):
    return (
        f"### Instruction:\n{example['instruction']}\n\n"
        f"### Input:\n{example['input']}\n\n"
        f"### Response:\n{example['output']}{tokenizer.eos_token}"
    )

def tokenize_fn(examples):
    texts = [
        format_example({"instruction": i, "input": inp, "output": o})
        for i, inp, o in zip(examples["instruction"], examples["input"], examples["output"])
    ]
    tokenized = tokenizer(texts, truncation=True, max_length=MAX_LENGTH, padding="max_length")
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

train_tokenized = train_ds.map(tokenize_fn, batched=True, remove_columns=train_ds.column_names)
val_tokenized = val_ds.map(tokenize_fn, batched=True, remove_columns=val_ds.column_names)

print(f"Tokenized: {len(train_tokenized)} train, {len(val_tokenized)} val")

## 7. Train

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

OUTPUT_DIR = "./fine_tuned_model"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=10,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    fp16=True,
    report_to="none",
    seed=42,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True),
    processing_class=tokenizer,
)

print("Starting training...")
trainer.train()
print("Training complete!")

## 8. Save and Test

In [ ]:
# Save the LoRA adapter
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save training config for the API server
config = {
    "base_model": BASE_MODEL,
    "lora_r": 16,
    "lora_alpha": 32,
    "epochs": 3,
}
with open(f"{OUTPUT_DIR}/training_config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"Model saved to {OUTPUT_DIR}")

# Keep reference for test cells
model = trainer.model

In [ ]:
# Quick test: generate SQL with the fine-tuned model
test_prompt = (
    "### Instruction:\nYou are a Snowflake SQL expert. Given a natural language question about "
    "the Olist Brazilian E-Commerce dataset, generate a correct Snowflake SQL query.\n"
    "Return ONLY the SQL query, no explanations.\n\n"
    "### Input:\nHow many orders are there in total?\n\n"
    "### Response:\n"
)

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=128, temperature=0.01, do_sample=False)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)
# Extract just the response
response = result.split("### Response:")[-1].strip()
print("Generated SQL:")
print(response)

In [ ]:
# Also test the BASE model (no LoRA) for comparison
from peft import PeftModel

# Disable LoRA adapters to get baseline output
model.disable_adapter_layers()

with torch.no_grad():
    outputs_base = model.generate(**inputs, max_new_tokens=128, temperature=0.01, do_sample=False)

result_base = tokenizer.decode(outputs_base[0], skip_special_tokens=True)
response_base = result_base.split("### Response:")[-1].strip()
print("Baseline (untrained) SQL:")
print(response_base)

# Re-enable LoRA
model.enable_adapter_layers()

## 9. Download Fine-Tuned Adapter

Download the adapter files and place them in `artifacts/fine_tuned_model/` in your repo.
Then run the API server locally:
```bash
python scripts/api_server.py --model-path artifacts/fine_tuned_model
```

## 9. Evaluation: Baseline vs Fine-Tuned (15 queries)

Runs 15 evaluation queries through both the **untrained baseline** and **fine-tuned LoRA** model.
Compares SQL quality, qualified table names, and syntactic correctness.

In [ ]:
import re, time, json

EVAL_QUERIES = [
    {"question": "How many orders are in the database?", "difficulty": "easy"},
    {"question": "What is the average review score?", "difficulty": "easy"},
    {"question": "How many sellers are in São Paulo state?", "difficulty": "easy"},
    {"question": "What is the total payment value for credit card payments?", "difficulty": "easy"},
    {"question": "How many distinct product categories are there?", "difficulty": "easy"},
    {"question": "What is the total revenue by customer state?", "difficulty": "medium"},
    {"question": "What are the top 5 product categories by number of orders?", "difficulty": "medium"},
    {"question": "What is the average delivery time in days by customer state?", "difficulty": "medium"},
    {"question": "How many orders per month were placed in 2018?", "difficulty": "medium"},
    {"question": "What is the average review score by product category?", "difficulty": "medium"},
    {"question": "What is the month-over-month growth rate in total payments?", "difficulty": "hard"},
    {"question": "Find the top 3 sellers by revenue in each state", "difficulty": "hard"},
    {"question": "What percentage of orders were delivered late?", "difficulty": "hard"},
    {"question": "Rank customer states by total number of orders", "difficulty": "hard"},
    {"question": "Calculate customer lifetime value and rank top 20 customers", "difficulty": "hard"},
]

def make_prompt(question):
    return (
        "### Instruction:\nYou are a Snowflake SQL expert. Given a natural language question about "
        "the Olist Brazilian E-Commerce dataset, generate a correct Snowflake SQL query.\n"
        "Use fully qualified table names: ANALYTICS_COPILOT.RAW.<TABLE>.\n"
        "Return ONLY the SQL query, no explanations.\n\n"
        f"### Input:\n{question}\n\n"
        "### Response:\n"
    )

def extract_sql(text):
    response = text.split("### Response:")[-1].strip()
    # Find the SQL statement
    match = re.search(r'\b(SELECT|WITH)\b', response, re.IGNORECASE)
    if match:
        response = response[match.start():]
    # Clean up trailing stuff
    response = re.split(r'\n\n[A-Z]', response)[0]
    response = response.split('\n\n###')[0]
    response = response.rstrip(';').strip()
    return response

def generate_sql(question, use_lora=True):
    if not use_lora:
        model.disable_adapter_layers()

    prompt = make_prompt(question)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    start = time.time()
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.01, do_sample=False)
    latency = (time.time() - start) * 1000

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    sql = extract_sql(text)

    if not use_lora:
        model.enable_adapter_layers()

    return sql, latency

# Run evaluation
results = []
print(f"{'#':<4} {'Difficulty':<10} {'Model':<12} {'Qualified':<10} {'Has SQL':<8} {'Latency':>8}")
print("-" * 70)

for i, q in enumerate(EVAL_QUERIES, 1):
    question = q["question"]
    difficulty = q["difficulty"]

    for model_name, use_lora in [("baseline", False), ("finetuned", True)]:
        sql, latency = generate_sql(question, use_lora=use_lora)
        has_sql = bool(sql and sql.strip())
        qualified = "ANALYTICS_COPILOT.RAW." in sql.upper() if sql else False

        results.append({
            "question": question,
            "difficulty": difficulty,
            "model": model_name,
            "sql_generated": sql,
            "has_sql": has_sql,
            "uses_qualified_names": qualified,
            "latency_ms": round(latency, 1),
        })

        print(f"{i:<4} {difficulty:<10} {model_name:<12} {'YES' if qualified else 'NO':<10} {'YES' if has_sql else 'NO':<8} {latency:>7.0f}ms")

print("\nEvaluation complete!")

In [ ]:
## 10. Download Fine-Tuned Adapter

Download the adapter files and place them in `artifacts/fine_tuned_model/` in your repo.
Then run the API server locally:
```bash
python scripts/api_server.py --model-path artifacts/fine_tuned_model
```

In [ ]:
# Zip and download the adapter
import shutil

shutil.make_archive("fine_tuned_model", "zip", OUTPUT_DIR)
files.download("fine_tuned_model.zip")
print("Download complete! Unzip into artifacts/fine_tuned_model/ in your repo.")